1. Filtrado del dataset de CHEMBL para usar el IC50 como indicador

In [ ]:
import pandas as pd

# 1. Cargar el dataset (los archivos de ChEMBL suelen usar ';' como separador)
df = pd.read_csv('Falcipain_2.csv', sep=';')

# 2. Mantener solo las filas que corresponden a mediones de IC50
df_ic50 = df[df['Standard Type'] == 'IC50'].copy()

# 3. Eliminar los registros que tengan el valor de IC50 ('Standard Value') como nulo
df_ic50 = df_ic50.dropna(subset=['Standard Value'])

# 4. Convertir los valores de nM a uM (dividiendo entre 1000) y actualizar las unidades
df_ic50['Standard Value'] = df_ic50['Standard Value'] / 1000.0
df_ic50['Standard Units'] = 'uM'

# 5. Eliminar columnas que hayan quedado completamente vacías (con todos sus valores nulos)
df_ic50 = df_ic50.dropna(axis=1, how='all')

# Opcional: Guardar el dataset limpio en un nuevo archivo CSV
df_ic50.to_csv('Falcipain_2_IC50.csv', index=False)

# Mostrar un resumen del resultado
#print(f"Número de filas finales: {len(df_ic50)}")
#print(df_ic50[['Molecule ChEMBL ID', 'Smiles', 'Standard Type', 'Standard Value', 'Standard Units']].head())

df_buscar = pd.read_csv('Falcipain_2_IC50.csv')
resultado= df_buscar[df_buscar['Molecule ChEMBL ID'] == 'CHEMBL4228926']
print(resultado)

2. Convertir el dataset de BindingBD a CSV

In [4]:
import pandas as pd
dataset_db = pd.read_csv('Falcipaina_2.tsv', sep='\t')
dataset_db.to_csv('Falcipaina_2.csv', index=False)

3. Unificación y deduplicación de compuestos contra Falcipain-2

In [ ]:
"""
Combina BindingBD.csv, Falcipain_2_IC50.csv (ChEMBL) y PubChem.csv
en un único dataset de compuestos ÚNICOS, usando el InChIKey
(identificador estándar de estructura química) como clave de
deduplicación en lugar de nombres o SMILES textuales (que varían
en escritura aunque representen la misma molécula).
"""

import time
import requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchiKey
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")


# ----------------------------------------------------------------
# 1. Utilidades de estructura química
# ----------------------------------------------------------------
def smiles_to_inchikey(smi):
    """Convierte un SMILES a InChIKey usando RDKit. Devuelve None si falla."""
    if not isinstance(smi, str) or not smi.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        return MolToInchiKey(mol)
    except Exception:
        return None


def fetch_smiles_from_cid(cid, session, cache):
    """Obtiene el SMILES canónico de un PubChem CID vía la API REST de PubChem."""
    if cid in cache:
        return cache[cid]
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/"
        f"{int(cid)}/property/CanonicalSMILES/TXT"
    )
    try:
        r = session.get(url, timeout=10)
        if r.status_code == 200:
            smi = r.text.strip()
            cache[cid] = smi
            return smi
    except Exception:
        pass
    cache[cid] = None
    return None


# ----------------------------------------------------------------
# 2. Cargar y procesar BindingDB
# ----------------------------------------------------------------
def load_bindingdb(path="BindingBD.csv"):
    df = pd.read_csv(path, low_memory=False)
    out = pd.DataFrame()
    out["SMILES"] = df["Ligand SMILES"]
    out["InChIKey"] = out["SMILES"].apply(smiles_to_inchikey)
    out["Nombre"] = df["BindingDB Ligand Name"]
    out["PubChem_CID"] = df["PubChem CID"]
    # BindingDB a veces reporta valores con prefijos '>' '<' -> quitarlos antes de convertir a número
    out["Ki_nM"] = pd.to_numeric(
        df["Ki (nM)"].astype(str).str.replace(r"[<>]", "", regex=True), errors="coerce"
    )
    out["IC50_nM"] = pd.to_numeric(
        df["IC50 (nM)"].astype(str).str.replace(r"[<>]", "", regex=True), errors="coerce"
    )
    out["Fuente"] = "BindingDB"
    return out


# ----------------------------------------------------------------
# 3. Cargar y procesar ChEMBL (Falcipain_2_IC50.csv)
# ----------------------------------------------------------------
def load_chembl(path="Falcipain_2_IC50.csv"):
    df = pd.read_csv(path, low_memory=False)
    out = pd.DataFrame()
    out["SMILES"] = df["Smiles"]
    out["InChIKey"] = out["SMILES"].apply(smiles_to_inchikey)
    out["Nombre"] = df["Molecule ChEMBL ID"]
    out["PubChem_CID"] = pd.NA
    out["Ki_nM"] = pd.NA
    # ChEMBL reporta en uM -> convertir a nM cuando el tipo es IC50
    ic50_um = df["Standard Value"].where(df["Standard Type"] == "IC50")
    out["IC50_nM"] = pd.to_numeric(ic50_um, errors="coerce") * 1000
    out["Fuente"] = "ChEMBL"
    out["ChEMBL_ID"] = df["Molecule ChEMBL ID"]
    return out


# ----------------------------------------------------------------
# 4. Cargar y procesar PubChem (sin SMILES -> hay que resolverlo por CID)
# ----------------------------------------------------------------
def load_pubchem(path="PubChem.csv", resolve_smiles=True, sleep=0.2):
    df = pd.read_csv(path, low_memory=False)
    out = pd.DataFrame()
    out["SMILES"] = pd.NA
    out["PubChem_CID"] = df["Compound_CID"]
    out["Nombre"] = df["Compound_Name"]
    out["Ki_nM"] = pd.NA
    ic50 = df["Activity_Value"].where(df["Activity_Type"] == "IC50")
    out["IC50_nM"] = pd.to_numeric(ic50, errors="coerce") * 1000  # uM -> nM
    out["Fuente"] = "PubChem"

    if resolve_smiles:
        cache = {}
        session = requests.Session()
        unique_cids = out["PubChem_CID"].dropna().unique()
        print(f"Resolviendo SMILES para {len(unique_cids)} CIDs únicos de PubChem...")
        for i, cid in enumerate(unique_cids, 1):
            fetch_smiles_from_cid(cid, session, cache)
            if i % 25 == 0:
                print(f"  {i}/{len(unique_cids)}")
            time.sleep(sleep)  # ser educado con la API pública
        out["SMILES"] = out["PubChem_CID"].map(cache)

    out["InChIKey"] = out["SMILES"].apply(smiles_to_inchikey)
    return out


# ----------------------------------------------------------------
# 5. Unificar y deduplicar
# ----------------------------------------------------------------
def unify(resolve_pubchem_smiles=True):
    b = load_bindingdb()
    c = load_chembl()
    p = load_pubchem(resolve_smiles=resolve_pubchem_smiles)

    all_df = pd.concat([b, c, p], ignore_index=True)

    # Clave de deduplicación: InChIKey si existe; si no, PubChem_CID como respaldo
    all_df["clave"] = all_df["InChIKey"]
    sin_estructura = all_df["clave"].isna()
    all_df.loc[sin_estructura, "clave"] = (
        "CID_" + all_df.loc[sin_estructura, "PubChem_CID"].astype(str)
    )

    resumen_previo = all_df["Fuente"].value_counts()

    agg = (
        all_df.groupby("clave")
        .agg(
            InChIKey=("InChIKey", "first"),
            SMILES=("SMILES", "first"),
            Nombre=("Nombre", "first"),
            PubChem_CID=("PubChem_CID", "first"),
            Ki_nM_min=("Ki_nM", "min"),
            IC50_nM_min=("IC50_nM", "min"),
            Fuentes=("Fuente", lambda s: ";".join(sorted(set(s)))),
            n_registros_originales=("Fuente", "count"),
        )
        .reset_index(drop=True)
    )

    print("\n=== Resumen ===")
    print("Registros originales por fuente:")
    print(resumen_previo)
    print(f"\nTotal registros originales: {len(all_df)}")
    print(f"Total compuestos únicos tras deduplicar: {len(agg)}")
    print(f"Duplicados eliminados: {len(all_df) - len(agg)}")
    print("\nCompuestos únicos por combinación de fuentes:")
    print(agg["Fuentes"].value_counts())

    agg.to_csv("compuestos_unicos_falcipain2.csv", index=False)
    print("\nGuardado: compuestos_unicos_falcipain2.csv")
    return agg


if __name__ == "__main__":
    unify(resolve_pubchem_smiles=True)

Resolviendo SMILES para 476 CIDs únicos de PubChem...
  25/476
  50/476
  75/476
  100/476
  125/476
  150/476
  175/476
  200/476
  225/476
  250/476
  275/476
  300/476
  325/476
  350/476
  375/476
  400/476
  425/476
  450/476
  475/476

=== Resumen ===
Registros originales por fuente:
Fuente
PubChem      506
ChEMBL       474
BindingDB    366
Name: count, dtype: int64

Total registros originales: 1346
Total compuestos únicos tras deduplicar: 890
Duplicados eliminados: 456

Compuestos únicos por combinación de fuentes:
Fuentes
PubChem                     300
ChEMBL                      223
BindingDB;ChEMBL            109
BindingDB                    96
BindingDB;ChEMBL;PubChem     90
BindingDB;PubChem            47
ChEMBL;PubChem               25
Name: count, dtype: int64

Guardado: compuestos_unicos_falcipain2.csv


4. Etiquetado de los compuestos por actividad

In [5]:
import pandas as pd

df = pd.read_csv('compuestos_unicos_falcipain2.csv')

df['Label'] = (df['IC50_nM_min'] < 1000).astype(int)

# 3. Guardar el dataset final

df= df.rename(columns={'IC50_nM_min':'IC50_nm'})
df.to_csv('ds_final.csv', index=False)

# Mostrar distribución de las clases
print(df['Label'].value_counts())

Label
0    701
1    189
Name: count, dtype: int64
